In [ ]:
## Setup — packages & environment

import sys, subprocess
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

required = ['pandas', 'numpy', 'scipy', 'matplotlib', 'seaborn', 'scikit-learn', 'networkx', 'openpyxl', 'reportlab']
for pkg in required:
    try:
        __import__(pkg.split('-')[0])
    except:
        install(pkg)

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
from scipy import stats
from sklearn.preprocessing import StandardScaler
import os, datetime as dt

RSEED = 2023
np.random.seed(RSEED)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
try:
    import ipykernel
    print('ipykernel installed')
except:
    print('ipykernel not installed')

In [ ]:
## 1. Theoretical Model Specification

# Hypothesized model:
# Motivation -> Course_Engagement -> Course_Performance
# Motivation -> Course_Performance (direct effect)
# Self_Efficacy -> Motivation

print('Theoretical Model:')
print('  Self_Efficacy -> Motivation -> Course_Engagement')
print('  Motivation -> Course_Performance')
print('  Course_Engagement -> Course_Performance')
print('  Self_Efficacy -> Course_Performance (direct)')

In [ ]:
## 2. Data Generation with Path Structure

np.random.seed(RSEED)
n = 400

# Exogenous variable
self_efficacy = np.random.normal(50, 10, n)

# Mediation pathway
motivation = 0.6 * self_efficacy + np.random.normal(0, 8, n)
course_engagement = 0.7 * motivation + np.random.normal(0, 8, n)

# Outcome with direct and indirect paths
course_performance = (
    0.3 * self_efficacy +
    0.4 * motivation +
    0.5 * course_engagement +
    np.random.normal(0, 10, n)
)

data = pd.DataFrame({
    'SelfEfficacy': self_efficacy,
    'Motivation': motivation,
    'CourseEngagement': course_engagement,
    'CoursePerformance': course_performance
})

# Standardize
data_std = (data - data.mean()) / data.std()

print('\nData Summary (standardized):')
print(data_std.head())
print(f'\nSample size: {len(data)}')

In [ ]:
## 3. Correlation and Regression Analysis

# Correlation matrix
corr_matrix = data_std.corr()
print('Correlation Matrix:')
print(corr_matrix.round(3))

# Simple regression analysis
from sklearn.linear_model import LinearRegression

# Direct paths
# Path: Self_Efficacy -> Motivation
X_se = data_std[['SelfEfficacy']].values
y_mot = data_std['Motivation'].values
model_se_mot = LinearRegression().fit(X_se, y_mot)
path_se_mot = model_se_mot.coef_[0]

# Path: Motivation -> Course_Engagement
X_mot = data_std[['Motivation']].values
y_eng = data_std['CourseEngagement'].values
model_mot_eng = LinearRegression().fit(X_mot, y_eng)
path_mot_eng = model_mot_eng.coef_[0]

# Total model on Course_Performance
X_all = data_std[['SelfEfficacy', 'Motivation', 'CourseEngagement']].values
y_perf = data_std['CoursePerformance'].values
model_full = LinearRegression().fit(X_all, y_perf)
path_se_perf = model_full.coef_[0]
path_mot_perf = model_full.coef_[1]
path_eng_perf = model_full.coef_[2]

print(f'\n\nPath Coefficients:')
print(f'SelfEfficacy -> Motivation: {path_se_mot:.3f}')
print(f'Motivation -> CourseEngagement: {path_mot_eng:.3f}')
print(f'SelfEfficacy -> CoursePerformance: {path_se_perf:.3f}')
print(f'Motivation -> CoursePerformance: {path_mot_perf:.3f}')
print(f'CourseEngagement -> CoursePerformance: {path_eng_perf:.3f}')

# Effect decomposition
indirect_se_perf = path_se_mot * path_mot_eng * path_eng_perf + path_se_mot * path_mot_perf
total_se_perf = path_se_perf + indirect_se_perf

print(f'\nEffect Decomposition (SelfEfficacy -> CoursePerformance):')
print(f'  Direct effect: {path_se_perf:.3f}')
print(f'  Indirect effect (via Motivation + Engagement): {indirect_se_perf:.3f}')
print(f'  Total effect: {total_se_perf:.3f}')

In [ ]:
## 4. Path Diagram Visualization

os.makedirs('figures', exist_ok=True)

# Create path diagram as network
G = nx.DiGraph()

# Add nodes
nodes = ['SelfEfficacy', 'Motivation', 'CourseEngagement', 'CoursePerformance']
for node in nodes:
    G.add_node(node)

# Add edges with weights (path coefficients)
edges = [
    ('SelfEfficacy', 'Motivation', path_se_mot),
    ('Motivation', 'CourseEngagement', path_mot_eng),
    ('SelfEfficacy', 'CoursePerformance', path_se_perf),
    ('Motivation', 'CoursePerformance', path_mot_perf),
    ('CourseEngagement', 'CoursePerformance', path_eng_perf)
]

for source, target, weight in edges:
    G.add_edge(source, target, weight=weight)

# Draw path diagram
fig, ax = plt.subplots(figsize=(12, 8))
pos = {
    'SelfEfficacy': (0, 2),
    'Motivation': (2, 2),
    'CourseEngagement': (4, 2),
    'CoursePerformance': (2, 0)
}

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_size=3000, node_color='lightblue', ax=ax, alpha=0.8)

# Draw edges with path coefficients
for source, target, data in G.edges(data=True):
    weight = data['weight']
    color = 'green' if weight > 0 else 'red'
    width = abs(weight) * 3
    nx.draw_networkx_edges(G, pos, [(source, target)], width=width, edge_color=color, 
                          ax=ax, alpha=0.6, arrowsize=20, arrowstyle='->')

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', ax=ax)

# Draw edge labels (path coefficients)
edge_labels = {(source, target): f'{data["weight"]:.2f}' for source, target, data in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=9, ax=ax)

ax.set_title('Structural Equation Model: Path Diagram', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('figures/01_path_diagram.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 01_path_diagram.png')

# 2. Correlation heatmap
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/02_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 02_correlation_heatmap.png')

# 3. Effect decomposition bar plot
fig, ax = plt.subplots(figsize=(9, 6))
effects = ['Direct', 'Indirect', 'Total']
values = [path_se_perf, indirect_se_perf, total_se_perf]
colors = ['steelblue', 'coral', 'green']
bars = ax.bar(effects, values, color=colors, edgecolor='black', alpha=0.8)
ax.set_ylabel('Effect Coefficient', fontsize=12)
ax.set_title('Effect Decomposition: SelfEfficacy on CoursePerformance', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/03_effect_decomposition.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: 03_effect_decomposition.png')

data_std.to_csv('sem_data.csv', index=False)
print('\nSaved SEM data')

In [ ]:
## 5. PDF Handout Generation

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, PageBreak
from reportlab.lib.enums import TA_JUSTIFY

pdf_path = 'Ch21_SEM_Handout.pdf'
doc = SimpleDocTemplate(pdf_path, pagesize=letter, rightMargin=0.75*inch, leftMargin=0.75*inch,
                        topMargin=0.75*inch, bottomMargin=0.75*inch)

styles = getSampleStyleSheet()
styleN = styles['Normal']
styleN.fontSize = 11
styleN.alignment = TA_JUSTIFY

story = []
story.append(Paragraph('<b>Chapter 21: Structural Equation Modeling</b>', styles['Heading1']))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>1. Introduction</b>', styles['Heading2']))
intro = (
    'Structural Equation Modeling (SEM) examines causal relationships through path analysis. '
    'Effects decompose into direct (unmediated) and indirect (mediated) pathways. '
    'SEM integrates measurement and structural models.'
)
story.append(Paragraph(intro, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>2. Theoretical Model</b>', styles['Heading2']))
theory = (
    'Self-efficacy influences motivation and course engagement, which jointly affect course performance. '
    'Self-efficacy also has a direct effect on performance, net of motivation and engagement.'
)
story.append(Paragraph(theory, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>3. Methods</b>', styles['Heading2']))
methods = (
    f'Data: {n} respondents. Regression analysis estimated path coefficients. '
    'Total, direct, and indirect effects calculated.'
)
story.append(Paragraph(methods, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>4. Results</b>', styles['Heading2']))
story.append(Spacer(1, 6))

try:
    if os.path.exists('figures/01_path_diagram.png'):
        story.append(Image('figures/01_path_diagram.png', width=520, height=400))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 1: Path diagram with standardized coefficients.', styleN))
        story.append(Spacer(1, 12))
except: pass

try:
    if os.path.exists('figures/03_effect_decomposition.png'):
        story.append(Image('figures/03_effect_decomposition.png', width=480, height=300))
        story.append(Spacer(1, 6))
        story.append(Paragraph('Fig 2: Effect decomposition showing mediation.', styleN))
        story.append(Spacer(1, 12))
except: pass

story.append(PageBreak())
story.append(Paragraph('<b>5. Key Findings</b>', styles['Heading2']))
findings = (
    f'Direct effect of Self-Efficacy on Performance: {path_se_perf:.3f}. '
    f'Indirect effect through Motivation and Engagement: {indirect_se_perf:.3f}. '
    f'Total effect: {total_se_perf:.3f}. '
    'Results suggest significant mediation pathways.'
)
story.append(Paragraph(findings, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph('<b>6. Limitations</b>', styles['Heading2']))
lim = (
    'Path analysis assumes linear relationships and single indicators per construct. '
    'Full SEM would include latent variable measurement models.'
)
story.append(Paragraph(lim, styleN))
story.append(Spacer(1, 12))

story.append(Paragraph(f'Generated on: {dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}', styleN))

try:
    doc.build(story)
    print(f'Saved PDF: {pdf_path}')
except Exception as e:
    print(f'PDF generation failed: {e}')